# Sua primeira análise em Python: os chamados do SP156

**Curso:** Análise de Dados para Decisões Estratégicas · Giselle Falcão Academy
**Módulo 1 · Prática 1.4** · Ferramenta: Google Colab

Que bom te ver por aqui! Este é o seu primeiro notebook — um documento que mistura **texto explicativo** (como esta célula) com **código executável**.

**Como usar:**
1. Passe o mouse sobre uma célula de código e clique no botão **▶** à esquerda (ou pressione `Ctrl+Enter`).
2. Execute as células **na ordem**, de cima para baixo.
3. Leia os comentários do código (as linhas que começam com `#`) — eles explicam cada passo.
4. Errou algo? Sem drama: menu **Ambiente de execução → Reiniciar sessão** e comece do topo.

> Friozinho na barriga por causa da palavra "Python"? Ótimo sinal: é assim que aprendizado se sente. — Giselle

## O contexto

O **SP156** é o canal oficial de atendimento ao cidadão da Prefeitura de São Paulo: buraco na via, poda de árvore, lâmpada apagada, remoção de entulho... Os registros são publicados no [Portal de Dados Abertos da Prefeitura](https://dados.prefeitura.sp.gov.br) (busque por "SP156").

**Pergunta de gestão desta prática:** *quais serviços a população mais solicita e onde o atendimento demora mais?*

Para começar sem tropeços, vamos usar uma **amostra didática de 30 chamados**, embutida neste notebook e inspirada no formato do dataset real. Dicionário de dados:

| Coluna | Tipo | Significado |
|--------|------|-------------|
| `id_chamado` | numérica (identificador) | número único do chamado |
| `data_abertura` | data | quando o cidadão abriu o chamado |
| `servico` | categórica | o que foi solicitado |
| `distrito` | categórica | região da cidade |
| `canal` | categórica | por onde o chamado entrou |
| `dias_para_resolucao` | numérica | dias até a conclusão |
| `status` | categórica | situação do chamado |

In [ ]:
# Importações: as "caixas de ferramentas" que vamos usar.
# pandas  -> trabalhar com tabelas de dados (o canivete suíço da análise)
# io      -> ler o texto embutido como se fosse um arquivo
# matplotlib -> desenhar gráficos
import io

import pandas as pd
import matplotlib.pyplot as plt

print('Ferramentas carregadas! Pode seguir para a próxima célula.')

In [ ]:
# Amostra didática no formato CSV (valores separados por vírgula).
# Na vida real, você baixaria este arquivo do Portal de Dados Abertos —
# a última célula do notebook mostra como.
dados_csv = '''id_chamado,data_abertura,servico,distrito,canal,dias_para_resolucao,status
1,2026-03-02,Buraco na via,Itaquera,Telefone,12,Concluído
2,2026-03-02,Poda de árvore,Pinheiros,Portal web,45,Concluído
3,2026-03-03,Buraco na via,Itaquera,App SP156,15,Concluído
4,2026-03-03,Lâmpada apagada,Sé,Telefone,5,Concluído
5,2026-03-04,Remoção de entulho,Capão Redondo,Telefone,30,Concluído
6,2026-03-05,Buraco na via,Capão Redondo,Telefone,28,Concluído
7,2026-03-05,Poda de árvore,Santana,Portal web,44,Concluído
8,2026-03-06,Buraco na via,Itaquera,Telefone,10,Concluído
9,2026-03-07,Lâmpada apagada,Mooca,App SP156,4,Concluído
10,2026-03-08,Remoção de entulho,Itaquera,Telefone,22,Concluído
11,2026-03-09,Buraco na via,Sé,Portal web,9,Concluído
12,2026-03-10,Poda de árvore,Capão Redondo,Telefone,60,Concluído
13,2026-03-11,Fiscalização de calçada,Pinheiros,Portal web,18,Concluído
14,2026-03-12,Buraco na via,Itaquera,App SP156,14,Concluído
15,2026-03-13,Lâmpada apagada,Itaquera,Telefone,6,Concluído
16,2026-03-14,Remoção de entulho,Capão Redondo,App SP156,35,Concluído
17,2026-03-16,Buraco na via,Mooca,Telefone,11,Concluído
18,2026-03-17,Poda de árvore,Itaquera,Telefone,48,Concluído
19,2026-03-18,Fiscalização de calçada,Sé,Portal web,20,Concluído
20,2026-03-19,Buraco na via,Capão Redondo,Telefone,31,Concluído
21,2026-03-20,Lâmpada apagada,Pinheiros,App SP156,3,Concluído
22,2026-03-21,Remoção de entulho,Itaquera,Telefone,25,Concluído
23,2026-03-23,Buraco na via,Santana,Portal web,13,Concluído
24,2026-03-24,Poda de árvore,Mooca,Telefone,55,Concluído
25,2026-03-25,Buraco na via,Itaquera,Telefone,16,Concluído
26,2026-03-26,Lâmpada apagada,Capão Redondo,Telefone,8,Concluído
27,2026-03-27,Remoção de entulho,Sé,App SP156,19,Concluído
28,2026-03-28,Fiscalização de calçada,Itaquera,Telefone,24,Concluído
29,2026-03-30,Buraco na via,Pinheiros,App SP156,7,Concluído
30,2026-03-31,Poda de árvore,Itaquera,Portal web,50,Concluído'''

# pd.read_csv lê o CSV e cria um DataFrame: a tabela do pandas.
# parse_dates avisa que 'data_abertura' é uma DATA, não um texto
# (lembra da Aula 1.3? tipo certo = análise certa).
df = pd.read_csv(io.StringIO(dados_csv), parse_dates=['data_abertura'])

print(f'Amostra carregada: {len(df)} chamados na tabela df.')

In [ ]:
# head() mostra as 5 primeiras linhas — o jeito clássico de "espiar" a tabela.
# Pergunta de ouro da Aula 1.3: cada linha desta tabela representa o quê?
df.head()

In [ ]:
# shape devolve (linhas, colunas); info() lista cada coluna, seu tipo
# e quantos valores NÃO nulos ela tem (compare: há faltantes aqui?).
print('Formato da tabela (linhas, colunas):', df.shape)
print()
df.info()

## Agregar: a gramática de toda análise

Ninguém decide olhando 30 linhas (muito menos 300 mil, que é a escala do SP156 real). A gente **agrega**: resume muitas linhas em poucas, seguindo sempre a mesma gramática:

> **"Calcule ISTO, agrupado por AQUILO."**

Em pandas, isso aparece de duas formas que você vai usar a vida toda:

- `value_counts()` — *conte as ocorrências de cada categoria*;
- `groupby(...)` — *agrupe por uma coluna e calcule algo em outra*.

Vamos às duas.

In [ ]:
# Quais serviços a população mais solicita?
# value_counts conta quantas vezes cada categoria aparece, já em ordem decrescente.
df['servico'].value_counts()

In [ ]:
# E por região? Contamos os chamados por distrito e desenhamos um gráfico de barras.
chamados_por_distrito = df['distrito'].value_counts()

chamados_por_distrito.plot(kind='barh', color='#6D28D9')  # roxo da casa :)
plt.title('Chamados por distrito — amostra SP156')
plt.xlabel('Número de chamados')
plt.ylabel('')
plt.gca().invert_yaxis()  # maior barra no topo, como num ranking
plt.tight_layout()
plt.show()

## O que isso diz para o gestor?

Repare no que as duas células acima acabaram de responder:

1. **"Buraco na via" lidera as solicitações** — é o serviço que mais mobiliza o cidadão nesta amostra.
2. **Itaquera concentra o maior volume de chamados** — se as equipes estão distribuídas por igual entre os distritos, este gráfico é um argumento para redistribuí-las.

Mas atenção (rigor acolhedor): volume não é a história toda. Um serviço pode ser **muito pedido e rápido de resolver**, e outro **pouco pedido e demoradíssimo**. Precisamos olhar o **tempo de resolução** — é a próxima célula.

In [ ]:
# Tempo médio de resolução por serviço:
# "calcule a MÉDIA de dias_para_resolucao, agrupada por servico".
tempo_medio_por_servico = (
    df.groupby('servico')['dias_para_resolucao']
      .mean()
      .sort_values(ascending=False)
      .round(1)
)

print('Tempo médio de resolução (dias) por serviço:')
print(tempo_medio_por_servico)

# Compare com o value_counts de serviços: o mais PEDIDO é o mais DEMORADO?

## 🏆 Desafio: agora é com você

Os padrões que você acabou de executar resolvem uma família enorme de perguntas. Prove isso respondendo a duas novas:

**(a)** Qual **canal de atendimento** é o mais usado? *(dica: é uma contagem de categoria — reveja a célula do `value_counts`)*

**(b)** Qual **distrito** tem o **maior tempo médio de resolução**? Gere também um **gráfico de barras** desse ranking. *(dica: é um `groupby` com média — reveja a célula anterior — seguido de um `.plot(kind='barh')`)*

Complete os `TODO` na célula abaixo. Errar e tentar de novo faz parte — o notebook não quebra.

In [ ]:
# ===== DESAFIO — complete os TODOs =====

# (a) Canal mais usado:
# TODO: conte as ocorrências de cada categoria da coluna 'canal'
# canais = df['...'].value_counts()
# print(canais)

# (b) Tempo médio de resolução por distrito (do maior para o menor):
# TODO: agrupe por 'distrito' e calcule a média de 'dias_para_resolucao'
# tempo_por_distrito = df.groupby('...')['...'].mean().sort_values(ascending=False)
# print(tempo_por_distrito)

# TODO: desenhe o gráfico de barras do ranking (b)
# tempo_por_distrito.plot(kind='barh', color='#0D9488')  # teal da casa
# plt.title('Tempo médio de resolução por distrito (dias)')
# plt.gca().invert_yaxis()
# plt.show()

# Quando terminar, escreva na célula de texto abaixo suas 2 frases de
# recomendação para o gestor — com base no que os números mostraram.

## Suas recomendações (edite esta célula!)

*Dê dois cliques aqui e escreva **duas frases de recomendação** para um gestor, começando com um verbo ("Priorizar...", "Realocar...", "Investigar..."). Use os números que você encontrou.*

1. ...
2. ...

---

### Próximos passos

- **Quer o dado real?** Busque "SP156" em https://dados.prefeitura.sp.gov.br, copie o link do CSV e carregue com `pd.read_csv(url, sep=';', encoding='latin-1')` — os parâmetros `sep` e `encoding` são as armadilhas 4 e 5 da Aula 1.3 em carne e osso. Os arquivos são grandes; comece por um semestre só.
- **Entregue a prática:** menu **Compartilhar → Qualquer pessoa com o link → Leitor**, e envie o link na plataforma.
- No **Módulo 2**, você refaz esse raciocínio no Google Sheets — e descobre que a gramática "calcule isto agrupado por aquilo" é a mesma em qualquer ferramenta.

Parabéns pela primeira análise. De verdade. — **Giselle Falcão**